In [1]:
# ============================================================
# COMBINE PCA ARTIFACTS — FINAL SUPPLEMENT TABLES
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".").resolve()
ARTIFACT_DIR = BASE_DIR / "final_pca_artifacts"
OUTPUT_DIR = BASE_DIR / "final_pca_variance_backprojection_PCA_trainonly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOMES = ["victimization", "perpetration", "overlap"]

def assign_domain(feature):
    f = str(feature).upper()

    if any(x in f for x in ["PAÍS", "PAIS", "ETNIA", "EDAD", "GENERO", "GÉNERO", "ORIENTSEX"]):
        return "Sociodemographic indicators"
    if "FUGAS" in f:
        return "Running away"
    if "ABUSOSUBS" in f or "SUBS" in f:
        return "Substance use"
    if "CONVIVEN" in f:
        return "Household composition"
    if "AUTOEFIC" in f:
        return "Self-efficacy"
    if "IMPULS" in f:
        return "Impulsivity"
    if "APOYO" in f:
        return "Social support"
    if "MORAL" in f:
        return "Moral domain"

    return "Other"

all_variance = []
all_loadings = []
all_top = []
all_feature_summary = []
all_domain_summary = []

for outcome in OUTCOMES:
    p = ARTIFACT_DIR / outcome

    variance = pd.read_csv(p / "pca_variance.csv")
    loadings = pd.read_csv(p / "pca_loadings.csv")
    top = pd.read_csv(p / "pca_top_loadings_by_component.csv")

    loadings["domain"] = loadings["feature"].apply(assign_domain)
    top["domain"] = top["feature"].apply(assign_domain)

    feature_summary = (
        loadings
        .assign(
            weighted_abs_loading=lambda d: d["abs_loading"] * d["explained_variance_ratio_component"]
        )
        .groupby(["outcome", "feature", "domain"], as_index=False)
        .agg(
            total_abs_loading=("abs_loading", "sum"),
            variance_weighted_abs_loading=("weighted_abs_loading", "sum"),
            max_abs_loading=("abs_loading", "max"),
        )
    )

    total_weighted = feature_summary["variance_weighted_abs_loading"].sum()

    feature_summary["normalized_variance_weighted_contribution"] = (
        feature_summary["variance_weighted_abs_loading"] / total_weighted
        if total_weighted > 0
        else np.nan
    )

    feature_summary = feature_summary.sort_values(
        ["outcome", "normalized_variance_weighted_contribution"],
        ascending=[True, False]
    )

    domain_summary = (
        feature_summary
        .groupby(["outcome", "domain"], as_index=False)
        .agg(
            variance_weighted_abs_loading=("variance_weighted_abs_loading", "sum"),
            normalized_variance_weighted_contribution=("normalized_variance_weighted_contribution", "sum"),
            n_features=("feature", "nunique"),
        )
        .sort_values(
            ["outcome", "normalized_variance_weighted_contribution"],
            ascending=[True, False]
        )
    )

    all_variance.append(variance)
    all_loadings.append(loadings)
    all_top.append(top)
    all_feature_summary.append(feature_summary)
    all_domain_summary.append(domain_summary)

variance_all = pd.concat(all_variance, ignore_index=True)
loadings_all = pd.concat(all_loadings, ignore_index=True)
top_all = pd.concat(all_top, ignore_index=True)
feature_summary_all = pd.concat(all_feature_summary, ignore_index=True)
domain_summary_all = pd.concat(all_domain_summary, ignore_index=True)

variance_all.to_csv(OUTPUT_DIR / "pca_variance_all_outcomes.csv", index=False)
loadings_all.to_csv(OUTPUT_DIR / "pca_loadings_all_outcomes.csv", index=False)
top_all.to_csv(OUTPUT_DIR / "pca_top_loadings_by_component_all_outcomes.csv", index=False)
feature_summary_all.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_all_outcomes.csv", index=False)
domain_summary_all.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_all_outcomes.csv", index=False)

# Pretty
variance_pretty = variance_all.copy()
variance_pretty["explained_variance_percent"] = variance_pretty["explained_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty["cumulative_variance_percent"] = variance_pretty["cumulative_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty.to_csv(OUTPUT_DIR / "pca_variance_all_outcomes_pretty.csv", index=False)

feature_pretty = feature_summary_all.copy()
feature_pretty["normalized_variance_weighted_contribution_percent"] = (
    feature_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
feature_pretty.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_all_outcomes_pretty.csv", index=False)

domain_pretty = domain_summary_all.copy()
domain_pretty["normalized_variance_weighted_contribution_percent"] = (
    domain_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
domain_pretty.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_all_outcomes_pretty.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print("\n=== PCA VARIANCE SUMMARY ===")
summary = (
    variance_all
    .groupby("outcome")
    .agg(
        n_components=("component_number", "max"),
        retained_variance_percent=("explained_variance_percent", "sum"),
        final_cumulative_percent=("cumulative_variance_percent", "max")
    )
    .reset_index()
)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n=== TOP 10 FEATURES BY VARIANCE-WEIGHTED BACK-PROJECTION ===")
for outcome in OUTCOMES:
    print("\n", outcome)
    print(
        feature_summary_all[feature_summary_all["outcome"] == outcome]
        .head(10)
        [["feature", "domain", "normalized_variance_weighted_contribution"]]
        .to_string(index=False, float_format=lambda x: f"{x:.4f}")
    )

print("\n=== DOMAIN SUMMARY ===")
print(domain_summary_all.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSaved outputs to:")
print(OUTPUT_DIR.resolve())


=== PCA VARIANCE SUMMARY ===
      outcome  n_components  retained_variance_percent  final_cumulative_percent
      overlap            27                     100.00                    100.00
 perpetration            27                     100.00                    100.00
victimization            27                     100.00                    100.00

=== TOP 10 FEATURES BY VARIANCE-WEIGHTED BACK-PROJECTION ===

 victimization
      feature                      domain  normalized_variance_weighted_contribution
   GENERO.BN1 Sociodemographic indicators                                     0.0943
   GENERO.BN0 Sociodemographic indicators                                     0.0943
   CONVIVEN_H       Household composition                                     0.0747
      PORNO.T                       Other                                     0.0643
   CONVIVEN.2       Household composition                                     0.0642
     ETNIA.BN Sociodemographic indicators                 

In [2]:
# ============================================================
# CORRECT PCA SUPPLEMENT TABLES — RETAINED COMPONENTS ONLY
# PCA TRAIN-ONLY FINAL VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".").resolve()
ARTIFACT_DIR = BASE_DIR / "final_pca_artifacts"
OUTPUT_DIR = BASE_DIR / "final_pca_variance_backprojection_PCA_trainonly_RETAINED_ONLY"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETAINED_COMPONENTS = {
    "victimization": 18,
    "perpetration": 22,
    "overlap": 18,
}

OUTCOMES = ["victimization", "perpetration", "overlap"]


def assign_domain(feature):
    f = str(feature).upper()

    if any(x in f for x in ["PAÍS", "PAIS", "ETNIA", "EDAD", "GENERO", "GÉNERO", "ORIENTSEX"]):
        return "Sociodemographic indicators"
    if "FUGAS" in f:
        return "Running away"
    if "ABUSOSUBS" in f or "SUBS" in f:
        return "Substance use"
    if "CONVIVEN" in f:
        return "Household composition"
    if "AUTOEFIC" in f:
        return "Self-efficacy"
    if "IMPULS" in f:
        return "Impulsivity"
    if "APOYO" in f:
        return "Social support"
    if "MORAL" in f:
        return "Moral domain"
    if "PORNO" in f:
        return "Pornography exposure"

    return "Other"


all_variance = []
all_loadings = []
all_top = []
all_feature_summary = []
all_domain_summary = []

for outcome in OUTCOMES:
    n_keep = RETAINED_COMPONENTS[outcome]
    p = ARTIFACT_DIR / outcome

    variance = pd.read_csv(p / "pca_variance.csv")
    loadings = pd.read_csv(p / "pca_loadings.csv")

    # Keep only retained components
    variance_ret = variance[variance["component_number"] <= n_keep].copy()
    loadings_ret = loadings[loadings["component_number"] <= n_keep].copy()

    # Recompute cumulative variance within original PCA sequence
    variance_ret["retained_component"] = True
    variance_ret["retained_n_components"] = n_keep
    variance_ret["retained_variance_percent_total"] = variance_ret["explained_variance_percent"].sum()

    loadings_ret["domain"] = loadings_ret["feature"].apply(assign_domain)

    # Top loadings by retained component
    top_ret = (
        loadings_ret
        .sort_values(["component_number", "abs_loading"], ascending=[True, False])
        .groupby("component_number")
        .head(8)
        .reset_index(drop=True)
    )
    top_ret["domain"] = top_ret["feature"].apply(assign_domain)

    # Approximate variance-weighted back-projection, retained components only
    feature_summary = (
        loadings_ret
        .assign(
            weighted_abs_loading=lambda d: d["abs_loading"] * d["explained_variance_ratio_component"]
        )
        .groupby(["outcome", "feature", "domain"], as_index=False)
        .agg(
            total_abs_loading=("abs_loading", "sum"),
            variance_weighted_abs_loading=("weighted_abs_loading", "sum"),
            max_abs_loading=("abs_loading", "max"),
        )
    )

    total_weighted = feature_summary["variance_weighted_abs_loading"].sum()

    feature_summary["normalized_variance_weighted_contribution"] = (
        feature_summary["variance_weighted_abs_loading"] / total_weighted
        if total_weighted > 0
        else np.nan
    )

    feature_summary = feature_summary.sort_values(
        ["outcome", "normalized_variance_weighted_contribution"],
        ascending=[True, False]
    )

    domain_summary = (
        feature_summary
        .groupby(["outcome", "domain"], as_index=False)
        .agg(
            variance_weighted_abs_loading=("variance_weighted_abs_loading", "sum"),
            normalized_variance_weighted_contribution=("normalized_variance_weighted_contribution", "sum"),
            n_features=("feature", "nunique"),
        )
        .sort_values(
            ["outcome", "normalized_variance_weighted_contribution"],
            ascending=[True, False]
        )
    )

    all_variance.append(variance_ret)
    all_loadings.append(loadings_ret)
    all_top.append(top_ret)
    all_feature_summary.append(feature_summary)
    all_domain_summary.append(domain_summary)


variance_all = pd.concat(all_variance, ignore_index=True)
loadings_all = pd.concat(all_loadings, ignore_index=True)
top_all = pd.concat(all_top, ignore_index=True)
feature_summary_all = pd.concat(all_feature_summary, ignore_index=True)
domain_summary_all = pd.concat(all_domain_summary, ignore_index=True)

# Save full corrected outputs
variance_all.to_csv(OUTPUT_DIR / "pca_variance_retained_components_only.csv", index=False)
loadings_all.to_csv(OUTPUT_DIR / "pca_loadings_retained_components_only.csv", index=False)
top_all.to_csv(OUTPUT_DIR / "pca_top_loadings_by_component_retained_only.csv", index=False)
feature_summary_all.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_retained_only.csv", index=False)
domain_summary_all.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_retained_only.csv", index=False)

# Pretty
variance_pretty = variance_all.copy()
variance_pretty["explained_variance_percent"] = variance_pretty["explained_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty["cumulative_variance_percent"] = variance_pretty["cumulative_variance_percent"].map(lambda x: f"{x:.2f}%")
variance_pretty["retained_variance_percent_total"] = variance_pretty["retained_variance_percent_total"].map(lambda x: f"{x:.2f}%")
variance_pretty.to_csv(OUTPUT_DIR / "pca_variance_retained_components_only_pretty.csv", index=False)

feature_pretty = feature_summary_all.copy()
feature_pretty["normalized_variance_weighted_contribution_percent"] = (
    feature_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
feature_pretty.to_csv(OUTPUT_DIR / "pca_feature_backprojection_summary_retained_only_pretty.csv", index=False)

domain_pretty = domain_summary_all.copy()
domain_pretty["normalized_variance_weighted_contribution_percent"] = (
    domain_pretty["normalized_variance_weighted_contribution"] * 100
).map(lambda x: f"{x:.2f}%")
domain_pretty.to_csv(OUTPUT_DIR / "pca_domain_backprojection_summary_retained_only_pretty.csv", index=False)

# Display
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print("\n=== PCA VARIANCE SUMMARY — RETAINED COMPONENTS ONLY ===")
summary = (
    variance_all
    .groupby("outcome")
    .agg(
        n_components=("component_number", "max"),
        retained_variance_percent=("explained_variance_percent", "sum"),
        final_cumulative_percent=("cumulative_variance_percent", "max")
    )
    .reset_index()
)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n=== TOP 10 FEATURES BY VARIANCE-WEIGHTED BACK-PROJECTION — RETAINED ONLY ===")
for outcome in OUTCOMES:
    print("\n", outcome)
    print(
        feature_summary_all[feature_summary_all["outcome"] == outcome]
        .head(10)
        [["feature", "domain", "normalized_variance_weighted_contribution"]]
        .to_string(index=False, float_format=lambda x: f"{x:.4f}")
    )

print("\n=== DOMAIN SUMMARY — RETAINED ONLY ===")
print(domain_summary_all.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSaved outputs to:")
print(OUTPUT_DIR.resolve())


=== PCA VARIANCE SUMMARY — RETAINED COMPONENTS ONLY ===
      outcome  n_components  retained_variance_percent  final_cumulative_percent
      overlap            18                      95.00                     95.00
 perpetration            22                      99.09                     99.09
victimization            18                      95.08                     95.08

=== TOP 10 FEATURES BY VARIANCE-WEIGHTED BACK-PROJECTION — RETAINED ONLY ===

 victimization
      feature                      domain  normalized_variance_weighted_contribution
   GENERO.BN1 Sociodemographic indicators                                     0.0956
   GENERO.BN0 Sociodemographic indicators                                     0.0956
   CONVIVEN_H       Household composition                                     0.0757
      PORNO.T        Pornography exposure                                     0.0650
   CONVIVEN.2       Household composition                                     0.0650
     ETNIA.BN S